In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
import numpy as np

# Datos de entrenamiento
texts = [
    "open browser", "open the browser", "open chrome", "open crome", "search in chrome", "begin chrome", "start chrome",
    "close browser", "close the browser", "close chrome", "close crome", "end chrome", "finish chrome",
    "open vlc", "open music player", "reproduce music", "reproduce music in vlc", "open v lc", "open v l c", "turn the music up",
]
labels = [
    "open_browser", "open_browser", "open_browser", "open_browser", "open_browser", "open_browser", "open_browser",
    "close_browser", "close_browser", "close_browser", "close_browser", "close_browser", "close_browser",
    "open_music_player", "open_music_player", "open_music_player", "open_music_player", "open_music_player", "open_music_player", "open_music_player",
]

# Tokenización y preparación de datos
tokenizer = Tokenizer(num_words=1000, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
padded_sequences = pad_sequences(sequences, padding='post')

# Convertir etiquetas a números
label_mapping = {label: i for i, label in enumerate(set(labels))}
y_train = np.array([label_mapping[label] for label in labels])

# Crear modelo
model = keras.Sequential([
    keras.layers.Embedding(1000, 16, input_length=len(padded_sequences[0])),
    keras.layers.GlobalAveragePooling1D(),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(len(label_mapping), activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(padded_sequences, y_train, epochs=30)

# Guardar modelo y tokenizer
model.save("liz_0001.keras")
import pickle
with open("tokenizer.pickle", "wb") as handle:
    pickle.dump(tokenizer, handle)
with open("label_mapping.pickle", "wb") as handle:
    pickle.dump(label_mapping, handle)

Epoch 1/30
1/1 [==============================] - 1s 1s/step - loss: 1.0967 - accuracy: 0.3333
Epoch 2/30
1/1 [==============================] - 0s 7ms/step - loss: 1.0948 - accuracy: 0.4444
Epoch 3/30
1/1 [==============================] - 0s 9ms/step - loss: 1.0929 - accuracy: 0.6111
Epoch 4/30
1/1 [==============================] - 0s 9ms/step - loss: 1.0912 - accuracy: 0.6111
Epoch 5/30
1/1 [==============================] - 0s 8ms/step - loss: 1.0895 - accuracy: 0.6111
Epoch 6/30
1/1 [==============================] - 0s 12ms/step - loss: 1.0878 - accuracy: 0.6667
Epoch 7/30
1/1 [==============================] - 0s 5ms/step - loss: 1.0861 - accuracy: 0.7222
Epoch 8/30
1/1 [==============================] - 0s 7ms/step - loss: 1.0845 - accuracy: 0.7222
Epoch 9/30
1/1 [==============================] - 0s 12ms/step - loss: 1.0828 - accuracy: 0.7222
Epoch 10/30
1/1 [==============================] - 0s 9ms/step - loss: 1.0810 - accuracy: 0.7222
Epoch 11/30
1/1 [=====================

In [58]:
model = tf.keras.models.load_model("liz_0001.keras")
with open("tokenizer.pickle", "rb") as handle:
    tokenizer = pickle.load(handle)
with open("label_mapping.pickle", "rb") as handle:
    label_mapping = pickle.load(handle)

In [59]:
import os

# Funciones para ejecutar comandos
def open_browser(): os.system("start chrome")
def close_browser(): os.system("taskkill /IM chrome.exe /F")
def open_music_player(): os.system("start vlc")

actions = {
    "open_browser": open_browser,
    "close_browser": close_browser,
    "open_music_player": open_music_player
}

In [60]:
def get_intent(text):
    sequence = tokenizer.texts_to_sequences([text])
    padded = tf.keras.preprocessing.sequence.pad_sequences(sequence, maxlen=model.input_shape[1], padding='post')
    print(padded)
    prediction = model.predict(padded)
    print(prediction)
    intent = list(label_mapping.keys())[np.argmax(prediction)]
    return intent

In [61]:
import pyaudio
import vosk
import json

def escuchar():
    model = vosk.Model("./voice_engine/models/vosk-model-en-us-0.42-gigaspeech")
    recognizer = vosk.KaldiRecognizer(model, 16000)

    audio = pyaudio.PyAudio()
    stream = audio.open(format=pyaudio.paInt16, channels=1, rate=16000, input=True, frames_per_buffer=8192)
    stream.start_stream()

    print("🎤 Escuchando...")
    while True:
        data = stream.read(4096, exception_on_overflow=False)
        if recognizer.AcceptWaveform(data):
            result = json.loads(recognizer.Result())
            return result["text"]
        
def assistant():
    while True:
        command = escuchar()
        print(f"👤 Tú: {command}")
        if command:
            intent = get_intent(command)
            print(f"🤖 Intención detectada: {intent}")
            actions.get(intent, lambda: print("No entiendo ese comando"))()

In [52]:
assistant()

🎤 Escuchando...
👤 Tú: open chrome
[[2 4 0 0]]
1/1 [==============================] - 0s 264ms/step
[[0.33808053 0.3123912  0.34952834]]
🤖 Intención detectada: open_browser
🎤 Escuchando...
👤 Tú: open the listen
[[2 7 1 0]]
1/1 [==============================] - 0s 35ms/step
[[0.34046024 0.31290027 0.34663954]]
🤖 Intención detectada: open_browser
🎤 Escuchando...
👤 Tú: open vnc
[[2 1 0 0]]
1/1 [==============================] - 0s 52ms/step
[[0.3444546  0.30990687 0.34563857]]
🤖 Intención detectada: open_browser
🎤 Escuchando...
👤 Tú: took utopia o l i want to say
[[1 1 1 1]]
1/1 [==============================] - 0s 28ms/step
[[0.35014978 0.318409   0.3314412 ]]
🤖 Intención detectada: open_music_player
🎤 Escuchando...
👤 Tú: nobody who
[[1 1 0 0]]
1/1 [==============================] - 0s 29ms/step
[[0.34214285 0.32074097 0.33711615]]
🤖 Intención detectada: open_music_player
🎤 Escuchando...
👤 Tú: open music layer
[[2 6 1 0]]
1/1 [==============================] - 0s 34ms/step
[[0.3588098  

KeyboardInterrupt: 